## Original Code Issues: Before vs After

| Area | Original issue | Why it failed | Current fix |
| --- | --- | --- | --- |
| TokenEmbedding | `vacab_size` and `pading_idx` were misspelled, and `forward` was missing. | Calling `self.TokenEmbedding(x)` needs a real module forward pass. | Use `vocab_size`, `padding_idx`, and return the token embedding in `forward`. |
| PositionalEmbedding | Only `encoding[0, ...]` was filled. | That only writes one position instead of every sequence position. | Fill `encoding[:, 0::2]` and `encoding[:, 1::2]`, then reshape to `[1, max_len, d_model]`. |
| Embedding wrapper | The wrapper mixed class names and tensor logic. | Token embedding, position embedding, scaling, and dropout were not cleanly separated. | `TransformerEmbedding` now owns token + position + dropout. |
| MultiHeadAttention | The main idea was right, but there was no divisibility check. | If `d_model` is not divisible by `num_heads`, `view` will fail. | Add an assert and split the head reshape into `split_heads`. |
| LayerNorm | `gamma/beta` were defined, but `gama/bata` were used. | That raises `AttributeError` during forward. | Use the same parameter names: `self.gamma` and `self.beta`. |
| Decoder layer | Decoder reused `norm2` and only had two norms. | Decoder has three sublayers: self-attn, cross-attn, and ffn. | Use `norm1`, `norm2`, `norm3` with matching dropouts. |
| Transformer assembly | `super(...).init__` was misspelled, `scr/src` names were mixed, and single layers were used like full stacks. | Constructor args, embeddings, masks, and layer stacks did not line up. | Split into `EncoderLayer`/`DecoderLayer`, stack them in `Encoder`/`Decoder`, then assemble in `Transformer`. |
| Masks | Attention accepted masks, but the full model did not create them. | The decoder could attend to future tokens, and padding tokens could affect attention. | Add `make_src_mask` for padding and `make_tgt_mask` for padding plus future-token blocking. |

Main idea: your smaller pieces were close. The missing part was interface alignment: define one layer first, stack layers next, then let the top-level `Transformer` handle embeddings, masks, encoder, decoder, and final vocabulary projection.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

from torch import Tensor


# Before: this class had no forward method, so the wrapper could not call it like a layer.
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, padding_idx=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.d_model = d_model

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)


# Before: only position 0 was filled; positional encoding must be built for all positions.
class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len, device):
        super().__init__()

        # shape: [1, max_len, d_model]，前面的 1 用来和 batch 维度广播相加
        encoding = torch.zeros(max_len, d_model, device=device)
        position = torch.arange(0, max_len, device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, device=device) * (-math.log(10000.0) / d_model))

        encoding[:, 0::2] = torch.sin(position * div_term)
        encoding[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("encoding", encoding.unsqueeze(0))#这是在模型中注册一个持久缓冲区，表示这个变量不需要梯度更新，但会随着模型一起保存和加载

    def forward(self, x):
        # x: [batch_size, seq_len]
        seq_len = x.size(1)
        return self.encoding[:, :seq_len, :]


# This wrapper is the clean place to combine token embedding, position embedding, scaling, and dropout.
class TransformerEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, dropout, device, padding_idx=1):
        super().__init__()
        self.token_embedding = TokenEmbedding(vocab_size, d_model, padding_idx)
        self.positional_embedding = PositionalEmbedding(d_model, max_len, device)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        token_emb = self.token_embedding(x)
        pos_emb = self.positional_embedding(x)
        return self.dropout(token_emb + pos_emb)


# 保留你原来的类名习惯，后面如果已经写了 Embedding(...) 也还能用
Embedding = TransformerEmbedding


In [2]:
# Before: the attention idea was right; the risky part was hidden shape assumptions during head splitting.
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_combine = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(0.1)

    def split_heads(self, x):
        batch_size, seq_len, _ = x.size()
        # [batch, seq_len, d_model] -> [batch, num_heads, seq_len, d_k]
        return x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)

        q = self.split_heads(self.W_q(q))
        k = self.split_heads(self.W_k(k))
        v = self.split_heads(self.W_v(v))

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            # mask 会广播到 [batch, num_heads, query_len, key_len]
            scores = scores.masked_fill(mask == 0, -1e9)

        attn = torch.softmax(scores, dim=-1)
        context = torch.matmul(self.dropout(attn), v)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_combine(context)


# 兼容原来拼写成 MutiHeadAttention 的代码
# 兼容原来拼写成 MutiHeadAttention 的代码


In [3]:
# Before: gamma/beta were defined but misspelled in forward as gama/bata.
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x + self.beta


# 兼容原来的类名
Layernorm = LayerNorm


In [4]:
# One EncoderLayer is only one block; the full Encoder below stacks several of these blocks.
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ffn, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ffn),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ffn, d_model),
        )
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        attn_output = self.self_attn(x, x, x, src_mask)
        x = self.norm1(x + self.dropout1(attn_output))
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_output))
        return x


# DecoderLayer has three sublayers: masked self-attn, cross-attn, then feed-forward.
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ffn, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ffn),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ffn, d_model),
        )
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.norm3 = LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        self_attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(self_attn_output))

        cross_attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout2(cross_attn_output))

        ffn_output = self.ffn(x)
        x = self.norm3(x + self.dropout3(ffn_output))
        return x


class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ffn, n_layers, dropout, max_len, device, padding_idx=1):
        super().__init__()
        self.embedding = TransformerEmbedding(vocab_size, d_model, max_len, dropout, device, padding_idx)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ffn, dropout)
            for _ in range(n_layers)
        ])

    def forward(self, src, src_mask=None):
        x = self.embedding(src)
        for layer in self.layers:
            x = layer(x, src_mask)
        return x


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ffn, n_layers, dropout, max_len, device, padding_idx=1):
        super().__init__()
        self.embedding = TransformerEmbedding(vocab_size, d_model, max_len, dropout, device, padding_idx)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ffn, dropout)
            for _ in range(n_layers)
        ])

    def forward(self, tgt, enc_output, src_mask=None, tgt_mask=None):
        x = self.embedding(tgt)
        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return x


In [ ]:
# The top-level Transformer only assembles parts: masks, encoder, decoder, and output projection.
class Transformer(nn.Module):
    def __init__(
        self,
        src_pad_idx,
        trg_pad_idx,
        enc_voc_size,
        dec_voc_size,
        d_model=512,
        nhead=8,
        ffn_hidden=2048,
        n_layers=6,
        drop_prob=0.1,
        max_len=5000,
        device="cpu",
    ):
        super().__init__()
        self.device = torch.device(device)
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx

        self.encoder = Encoder(
            enc_voc_size, d_model, nhead, ffn_hidden, n_layers, drop_prob, max_len, self.device, src_pad_idx
        )
        self.decoder = Decoder(
            dec_voc_size, d_model, nhead, ffn_hidden, n_layers, drop_prob, max_len, self.device, trg_pad_idx
        )
        self.fc_out = nn.Linear(d_model, dec_voc_size)
    '''这些mask的作用和具体内容长什么样子?
    src_mask 用于 encoder 的自注意力，遮住输入序列中的填充符位置；tgt_mask 用于 decoder 的自注意力，既遮住目标序列中的填充符位置，也遮住未来位置（即右侧位置）。
    '''
    def make_src_mask(self, src):
        # src: [batch, src_len] -> [batch, 1, 1, src_len]
        return (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)#src != self.src_pad_idx 是一个布尔张量，表示哪些位置不是填充符；unsqueeze 用来增加维度以适应后续计算。

    def make_tgt_mask(self, tgt):
        # padding mask: [batch, 1, 1, tgt_len]
        tgt_pad_mask = (tgt != self.trg_pad_idx).unsqueeze(1).unsqueeze(2)

        # subsequent mask: [1, 1, tgt_len, tgt_len]，遮住未来位置
        tgt_len = tgt.size(1)
        subsequent_mask = torch.tril(torch.ones((tgt_len, tgt_len), device=tgt.device)).bool()
        subsequent_mask = subsequent_mask.unsqueeze(0).unsqueeze(1)

        return tgt_pad_mask & subsequent_mask

    def forward(self, src_input, trg_input):
        # src_input: [batch, src_len]
        # trg_input: [batch, trg_len]，训练时通常传入右移后的目标序列
        src_mask = self.make_src_mask(src_input)
        tgt_mask = self.make_tgt_mask(trg_input)

        enc_output = self.encoder(src_input, src_mask)
        dec_output = self.decoder(trg_input, enc_output, src_mask, tgt_mask)
        return self.fc_out(dec_output)




In [6]:
# 一个最小冒烟测试：只检查维度能不能跑通
# 输出 shape 应该是 [batch_size, trg_len, dec_voc_size]
device = "cuda" if torch.cuda.is_available() else "cpu"

src_pad_idx = 1
trg_pad_idx = 1
enc_voc_size = 100
dec_voc_size = 120

model = Transformer(
    src_pad_idx=src_pad_idx,
    trg_pad_idx=trg_pad_idx,
    enc_voc_size=enc_voc_size,
    dec_voc_size=dec_voc_size,
    d_model=32,
    nhead=4,
    ffn_hidden=64,
    n_layers=2,
    drop_prob=0.1,
    max_len=50,
    device=device,
).to(device)

src = torch.tensor([
    [2, 5, 7, 9, 1, 1],
    [4, 6, 8, 3, 2, 1],
], device=device)

tgt = torch.tensor([
    [2, 10, 11, 1, 1],
    [2, 12, 13, 14, 1],
], device=device)

out = model(src, tgt)
print(out.shape)


torch.Size([2, 5, 120])
